In [0]:


from pyspark.sql import functions as F

silver_table = "fintech_lakehouse.silver.orders_current"

# Read the latest clean order data from Silver
silver_df = spark.table(silver_table)



# Creating a business-ready orders dataset

gold_orders = (
    silver_df
    .withColumn(
        "order_timestamp",
        F.to_timestamp("event_time")
    )
    .withColumn(
        "order_date",
        F.to_date("event_time")
    )
    .withColumn(
        "order_hour",
        F.hour(F.to_timestamp("event_time"))
    )
    .withColumn(
        "order_value_category",
        F.when(F.col("amount") < 100, "LOW")
        .when(F.col("amount") < 500, "MEDIUM")
        .otherwise("HIGH")
    )
    .select(
        "order_id",
        "customer_id",
        "status",
        "amount",
        "order_value_category",
        "order_timestamp",
        "order_date",
        "order_hour",
        "is_cancelled",
        "sequence_number",
        "silver_updated_at"
    )
)

(
    gold_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "fintech_lakehouse.gold.orders_business"
    )
)

print("Created Gold table: orders_business")


# Create daily business KPIs

daily_kpis = (
    gold_orders
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),

        F.countDistinct("customer_id").alias(
            "unique_customers"
        ),

        F.sum(
            F.when(
                ~F.col("is_cancelled"),
                1
            ).otherwise(0)
        ).alias("successful_orders"),

        F.sum(
            F.when(
                F.col("is_cancelled"),
                1
            ).otherwise(0)
        ).alias("cancelled_orders"),

        F.round(
            F.sum(
                F.when(
                    ~F.col("is_cancelled"),
                    F.col("amount")
                ).otherwise(0)
            ),
            2
        ).alias("total_revenue"),

        F.round(
            F.avg(
                F.when(
                    ~F.col("is_cancelled"),
                    F.col("amount")
                )
            ),
            2
        ).alias("average_order_value")
    )
    .withColumn(
        "cancellation_rate_percent",
        F.round(
            (
                F.col("cancelled_orders") /
                F.col("total_orders")
            ) * 100,
            2
        )
    )
)

(
    daily_kpis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "fintech_lakehouse.gold.daily_order_kpis"
    )
)

print("Created Gold table: daily_order_kpis")



# Create customer-value table

customer_kpis = (
    gold_orders
    .filter(~F.col("is_cancelled"))
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),

        F.round(
            F.sum("amount"),
            2
        ).alias("customer_lifetime_value"),

        F.round(
            F.avg("amount"),
            2
        ).alias("average_order_value"),

        F.min("order_timestamp").alias("first_order_time"),

        F.max("order_timestamp").alias("latest_order_time")
    )
)

(
    customer_kpis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "fintech_lakehouse.gold.customer_kpis"
    )
)

print("Created Gold table: customer_kpis")



# Display the clear final data

display(
    spark.table(
        "fintech_lakehouse.gold.orders_business"
    )
    .orderBy(F.col("order_timestamp").desc())
)


display(
    spark.table(
        "fintech_lakehouse.gold.daily_order_kpis"
    )
    .orderBy(F.col("order_date").desc())
)



display(
    spark.table(
        "fintech_lakehouse.gold.customer_kpis"
    )
    .orderBy(
        F.col("customer_lifetime_value").desc()
    )
)